In [1]:
# we need to run on a py file instead of a jupyter notebook otherwise multiprocessing will not work properly
from agent import Agent, pit
from model_files.SLPolicyValueGPU import SLPolicyValueNetwork
import torch
import chess
import sys


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1 = SLPolicyValueNetwork().to(device)
model1.load_state_dict(torch.load("SL_trained_stockfish_trained.pth", map_location=torch.device("cuda"))["model"])
agent = Agent(policy_value_network=model1, c_puct=0.25, dirichlet_alpha=0.3, dirichlet_epsilon=0.0)
# epsilon is set to 0 for no noise

In [5]:
board = chess.Board()
human_turn = 1
while not board.is_game_over():
    print(board, "\n")
    if human_turn == 1:
        move = input("enter a move in UCI format\n")
        if move == "q":
            sys.exit()
        try:
            board.push_uci(move)
            human_turn *= -1
        except:
            continue
    
    else:
        move = agent.select_move(game_state=board, num_simulations=1600, temperature=0, debug=True)
        print("\n")
        board.push_uci(move)
        human_turn *= -1

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R 



SystemExit: 

In [2]:
board = chess.Board()
human_turn = 1
agent.stockfish.set_depth(15)
moves = []
move = None
while not board.is_game_over():
    print(board, "\n")
    if human_turn == 1:
        agent.stockfish.set_fen_position(board.fen())
        move = agent.stockfish.get_best_move()
        board.push_uci(move)
        human_turn *= -1
    
    else:
        move = agent.select_move(game_state=board, num_simulations=1600, temperature=0, debug=True)
        print("\n")
        board.push_uci(move)
        human_turn *= -1
    moves.append(move)
    print(moves)

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R 

['e2e4']
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R 

move: e7e5, count: 422.0
move: c7c5, count: 294.0
move: e7e6, count: 220.0
move: c7c6, count: 166.0
move: g8f6, count: 101.0
final eval:  -0.16324924971352195


['e2e4', 'e7e5']
r n b q k b n r
p p p p . p p p
. . . . . . . .
. . . . p . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R 

['e2e4', 'e7e5', 'g1f3']
r n b q k b n r
p p p p . p p p
. . . . . . . .
. . . . p . . .
. . . . P . . .
. . . . . N . .
P P P P . P P P
R N B Q K B . R 

move: b8c6, count: 694.0
move: g8f6, count: 154.0
move: d7d6, count: 123.0
move: d7d5, count: 79.0
move: a7a6, count: 55.0
final eval:  -0.03231466244248627


['e2e4', 'e7e5', 'g1f3', 'b8c6']
r . b q k b n r
p p p p . p p p
. . n . . . . .
. . . . p . . .
. . . . P . 

In [ ]:
agent.agent_vs_stockfish(2, 3200, "pgn_files/demo_3200_sims_vs_depth16.pgn, 45")